# Viettel AI Race — `predict-v2` Run All (Kaggle)

Pipeline **precision-first**, thay cho pipeline sinh văn bản của lần nộp 01 (14.4255).

```text
GLiNER spans (ngưỡng riêng theo type)
  → Qwen corrector: TRIỆU_CHỨNG → CHẨN_ĐOÁN   (GPU)
  → Qwen consensus additions cho type không có candidate  (GPU, cần 2 teacher)
  → trim generic prefix + loại header
  → exact-alias linking (ICD-10 tiếng Việt TT06 + RxNorm), chỉ emit khi khớp duy nhất
  → assertions rỗng
  → validate → output.zip
```

**Vì sao đổi cách làm.** Scorer của BTC đếm mỗi concept thừa **hai lần** vào mẫu số
của cả ba thành phần. Nên precision đáng giá hơn recall, và candidate thừa còn đắt
hơn nữa: một concept sai mang 3 mã tốn `2×(3+1)=8` đơn vị mẫu số thay vì 2.

**Trước khi Run All:**

1. Settings → Accelerator: **GPU T4 x2** (khuyến nghị) hoặc **P100**.
2. Settings → Internet: **On** — để tải weights và RxNorm.
3. Nếu chưa attach weights: đặt HF token trong Add-ons → Secrets với tên
   `HF_TOKEN`.

**Không cần attach gì cả.** Notebook tự chứa: package `medical_coder`, bảng
ICD-10 tiếng Việt và bản test Vòng 1 đều được nhúng sẵn. Chỉ cần tải lên đúng
một tệp `.ipynb` này.

Attach Dataset input vẫn được và **luôn được ưu tiên** hơn bản nhúng — bắt buộc
làm vậy khi chạy trên private test của BTC.

## 1. Kiểm tra GPU và môi trường

In [ ]:
!nvidia-smi || echo "Không thấy GPU — pipeline vẫn chạy được trên CPU nhưng KHÔNG có bước corrector."

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

WORK = Path("/kaggle/working")
IS_KAGGLE = Path("/kaggle").exists()
print("kaggle:", IS_KAGGLE, "| python:", sys.version.split()[0])

try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print("  ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
except ImportError:
    print("torch chưa được cài")

### P100 (`sm_60`)

Wheel torch mặc định của Kaggle có thể không chứa kiến trúc `sm_60`. Cell dưới chỉ
cài lại torch khi phát hiện P100 **và** arch hiện tại thiếu `sm_60`. Sau khi cài
lại phải **Restart Session** rồi Run All lần nữa — không thể tráo binary torch
trong kernel đã import nó.

In [ ]:
NEEDS_RESTART = False
try:
    import torch
    if torch.cuda.is_available():
        caps = {torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())}
        arches = torch.cuda.get_arch_list()
        if (6, 0) in caps and not any("sm_60" in a for a in arches):
            print("P100 nhưng torch thiếu sm_60 — cài lại torch CUDA 12.6")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "torch==2.10.0", "--index-url",
                            "https://download.pytorch.org/whl/cu126"], check=False)
            NEEDS_RESTART = True
        else:
            print("torch arch OK:", [a for a in arches if a.startswith("sm_")])
except Exception as exc:
    print("bỏ qua kiểm tra arch:", exc)

if NEEDS_RESTART:
    print("\n>>> HÃY CHỌN 'Restart Session' RỒI RUN ALL LẠI <<<")

## 2. Cài đặt

In [ ]:
%%capture install_log
!python -m pip install -q "gliner>=0.2.13" "transformers>=4.51" accelerate bitsandbytes pydantic

In [ ]:
import importlib
for module in ("gliner", "transformers", "pydantic"):
    try:
        importlib.import_module(module)
        print("ok  ", module)
    except ImportError as exc:
        print("LỖI", module, exc)

import torch
assert torch.cuda.is_available() or True, "không có CUDA"
print("cuda sau khi cài:", torch.cuda.is_available())

## 3. Source code

12 module của package `medical_coder` được ghi thẳng ra đĩa bằng
`%%writefile`, không nén, không mã hoá. Đọc được, sửa được ngay tại chỗ: gặp lỗi
trên Kaggle thì sửa cell rồi Restart & Run All, khỏi phải dựng lại notebook ở máy
rồi tải lên.

Đây là đúng những module mà nhánh predict-v2 cần. `pipeline.py` cùng backend LLM
sinh văn bản của lần nộp 01 **không** có ở đây — chúng không được dùng, và đưa vào
chỉ tổ đặt 43 KB code chết trước mặt người đọc.

> Sửa cell nào thì phải **Restart Session** rồi chạy lại, vì `medical_coder` đã
> được import vào kernel.

In [ ]:
import pathlib
pathlib.Path("/kaggle/working/medical_coder_src/medical_coder").mkdir(parents=True, exist_ok=True)
print("thư mục source:", "/kaggle/working/medical_coder_src/medical_coder")

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/__init__.py
"""Clinical concept extraction for the Viettel AI Race."""

__version__ = "0.1.0"

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/models.py
from __future__ import annotations

from enum import Enum
from typing import Any

from pydantic import BaseModel, ConfigDict, Field


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class EntityType(str, Enum):
    SYMPTOM = "TRIỆU_CHỨNG"
    TEST_NAME = "TÊN_XÉT_NGHIỆM"
    TEST_RESULT = "KẾT_QUẢ_XÉT_NGHIỆM"
    DIAGNOSIS = "CHẨN_ĐOÁN"
    MEDICATION = "THUỐC"


class AssertionType(str, Enum):
    NEGATED = "isNegated"
    FAMILY = "isFamily"
    HISTORICAL = "isHistorical"


class ExtractedMention(StrictModel):
    """Mention returned by the LLM before deterministic offset alignment."""

    text: str = Field(min_length=1)
    type: EntityType
    assertions: list[AssertionType]
    start_hint: int = Field(
        ge=0,
        description="Estimated zero-based start position in the original text.",
    )


class ExtractionResponse(StrictModel):
    entities: list[ExtractedMention]


class CandidateRequest(StrictModel):
    entity_index: int = Field(ge=0)
    type: EntityType
    text: str = Field(min_length=1)
    context: str


class CandidatePrediction(StrictModel):
    entity_index: int = Field(ge=0)
    candidates: list[str]


class NormalizationResponse(StrictModel):
    mappings: list[CandidatePrediction]


class AlignedEntity(StrictModel):
    text: str
    type: EntityType
    assertions: list[AssertionType]
    position: tuple[int, int]
    candidates: list[str] = Field(default_factory=list)

    def to_submission_dict(self) -> dict[str, Any]:
        result: dict[str, Any] = {
            "text": self.text,
            "type": self.type.value,
        }
        if self.type in {EntityType.DIAGNOSIS, EntityType.MEDICATION}:
            result["candidates"] = self.candidates
        result["assertions"] = [item.value for item in self.assertions]
        result["position"] = [self.position[0], self.position[1]]
        return result

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/validation.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Iterable

from .models import AssertionType, EntityType


ICD10_RE = re.compile(r"^[A-Z][0-9]{2}(?:\.[0-9A-Z]{1,4})?$")
RXCUI_RE = re.compile(r"^[0-9]+$")
ALLOWED_TYPES = {item.value for item in EntityType}
ALLOWED_ASSERTIONS = {item.value for item in AssertionType}
CANDIDATE_TYPES = {EntityType.DIAGNOSIS.value, EntityType.MEDICATION.value}
ASSERTION_TYPES = {
    EntityType.SYMPTOM.value,
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}


class SubmissionValidationError(ValueError):
    pass


def sanitize_candidates(
    entity_type: EntityType,
    candidates: Iterable[str],
    max_candidates: int,
    allowlist: set[str] | None = None,
) -> list[str]:
    pattern = ICD10_RE if entity_type == EntityType.DIAGNOSIS else RXCUI_RE
    result: list[str] = []
    for value in candidates:
        candidate = str(value).strip().upper() if entity_type == EntityType.DIAGNOSIS else str(value).strip()
        if not pattern.fullmatch(candidate):
            continue
        if allowlist is not None and candidate not in allowlist:
            continue
        if candidate not in result:
            result.append(candidate)
        if len(result) >= max_candidates:
            break
    return result


def validate_submission_record(raw_text: str, entities: list[dict[str, Any]]) -> None:
    if not isinstance(entities, list):
        raise SubmissionValidationError("Top-level JSON must be a list")

    seen: set[tuple[int, int, str]] = set()
    previous_position: tuple[int, int] | None = None

    for index, entity in enumerate(entities):
        if not isinstance(entity, dict):
            raise SubmissionValidationError(f"Entity {index} must be an object")

        required = {"text", "type", "assertions", "position"}
        missing = required - set(entity)
        if missing:
            raise SubmissionValidationError(f"Entity {index} missing fields: {sorted(missing)}")

        entity_type = entity["type"]
        if entity_type not in ALLOWED_TYPES:
            raise SubmissionValidationError(f"Entity {index} has invalid type: {entity_type!r}")

        has_candidates = "candidates" in entity
        if (entity_type in CANDIDATE_TYPES) != has_candidates:
            raise SubmissionValidationError(
                f"Entity {index}: candidates field does not match type {entity_type}"
            )

        position = entity["position"]
        if (
            not isinstance(position, list)
            or len(position) != 2
            or not all(isinstance(value, int) and not isinstance(value, bool) for value in position)
        ):
            raise SubmissionValidationError(f"Entity {index} has invalid position")
        start, end = position
        if not (0 <= start < end <= len(raw_text)):
            raise SubmissionValidationError(
                f"Entity {index} position [{start}, {end}] is outside text length {len(raw_text)}"
            )
        if raw_text[start:end] != entity["text"]:
            raise SubmissionValidationError(
                f"Entity {index} text does not match raw_text[{start}:{end}]"
            )

        assertions = entity["assertions"]
        if not isinstance(assertions, list) or any(
            item not in ALLOWED_ASSERTIONS for item in assertions
        ):
            raise SubmissionValidationError(f"Entity {index} has invalid assertions")
        if entity_type not in ASSERTION_TYPES and assertions:
            raise SubmissionValidationError(
                f"Entity {index}: assertions are not allowed for {entity_type}"
            )
        if len(assertions) != len(set(assertions)):
            raise SubmissionValidationError(f"Entity {index} has duplicate assertions")

        if has_candidates:
            candidates = entity["candidates"]
            if not isinstance(candidates, list) or any(
                not isinstance(item, str) for item in candidates
            ):
                raise SubmissionValidationError(f"Entity {index} has invalid candidates")
            pattern = ICD10_RE if entity_type == EntityType.DIAGNOSIS.value else RXCUI_RE
            if any(not pattern.fullmatch(item) for item in candidates):
                raise SubmissionValidationError(f"Entity {index} has malformed candidate code")
            if len(candidates) != len(set(candidates)):
                raise SubmissionValidationError(f"Entity {index} has duplicate candidates")

        current = (start, end)
        if previous_position is not None and current < previous_position:
            raise SubmissionValidationError("Entities are not sorted by position")
        previous_position = current

        identity = (start, end, entity_type)
        if identity in seen:
            raise SubmissionValidationError(f"Duplicate entity at index {index}")
        seen.add(identity)


def validate_output_directory(input_dir: Path, output_dir: Path) -> list[str]:
    errors: list[str] = []
    input_files = sorted(input_dir.glob("*.txt"), key=lambda path: int(path.stem))
    expected_names = {f"{path.stem}.json" for path in input_files}
    actual_names = {path.name for path in output_dir.glob("*.json")}

    missing = sorted(expected_names - actual_names)
    extra = sorted(actual_names - expected_names)
    if missing:
        errors.append(f"Missing output files: {', '.join(missing)}")
    if extra:
        errors.append(f"Unexpected output files: {', '.join(extra)}")

    for input_path in input_files:
        output_path = output_dir / f"{input_path.stem}.json"
        if not output_path.exists():
            continue
        try:
            raw_text = input_path.read_text(encoding="utf-8")
            entities = json.loads(output_path.read_text(encoding="utf-8"))
            validate_submission_record(raw_text, entities)
        except (OSError, UnicodeError, json.JSONDecodeError, SubmissionValidationError) as exc:
            errors.append(f"{output_path.name}: {exc}")
    return errors

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/submission.py
"""Packaging and final checks for a submission.

Kept separate from :mod:`medical_coder.pipeline` on purpose: these helpers are
generic, but `pipeline` pulls in the whole generative LLM backend, so importing
them from there would drag ~43 KB of unrelated code into any consumer — notably
the predict-v2 notebook, which uses none of it.
"""
from __future__ import annotations

import zipfile
from pathlib import Path

from .validation import validate_output_directory


def create_submission_zip(output_dir: Path, zip_path: Path) -> None:
    """Write `output/<id>.json` members, then read the archive back to verify."""
    output_files = sorted(
        (path for path in output_dir.glob("*.json") if path.stem.isdigit()),
        key=lambda path: int(path.stem),
    )
    if not output_files:
        raise FileNotFoundError(f"No JSON files found in {output_dir}")
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = zip_path.with_suffix(zip_path.suffix + ".tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in output_files:
            archive.write(path, arcname=f"output/{path.name}")
    temporary.replace(zip_path)

    expected = [f"output/{path.name}" for path in output_files]
    with zipfile.ZipFile(zip_path, "r") as archive:
        if archive.namelist() != expected:
            raise RuntimeError(f"ZIP verification failed: {zip_path}")
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt ZIP member: {bad_member}")


def validate_all(input_dir: Path, output_dir: Path) -> None:
    errors = validate_output_directory(input_dir, output_dir)
    if errors:
        formatted = "\n".join(f"- {error}" for error in errors)
        raise RuntimeError(f"Output validation failed:\n{formatted}")

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/terminology.py
from __future__ import annotations

import csv
import hashlib
import json
import math
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable


_NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
_EMBEDDER_CACHE: dict[tuple[str, str], object] = {}


def normalize_term(value: str) -> str:
    """Normalize a term for retrieval only; never use this text for offsets."""

    decomposed = unicodedata.normalize("NFD", value.casefold())
    without_marks = "".join(
        character
        for character in decomposed
        if unicodedata.category(character) != "Mn"
    )
    return _NON_ALNUM_RE.sub(" ", without_marks).strip()


def _character_ngrams(value: str, size: int = 3) -> set[str]:
    compact = value.replace(" ", "_")
    if len(compact) <= size:
        return {compact} if compact else set()
    return {compact[index : index + size] for index in range(len(compact) - size + 1)}


@dataclass(frozen=True)
class TerminologyEntry:
    code: str
    label: str
    aliases: tuple[str, ...]

    @property
    def search_text(self) -> str:
        values = (self.label, *self.aliases)
        return " ; ".join(value for value in values if value)


@dataclass(frozen=True)
class RetrievedCandidate:
    code: str
    label: str
    score: float
    exact: bool


def _iter_jsonl(path: Path) -> Iterable[TerminologyEntry]:
    for line_number, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            code = str(item["code"]).strip()
            label = str(item["label"]).strip()
            raw_aliases = item.get("aliases", [])
            if isinstance(raw_aliases, str):
                raw_aliases = raw_aliases.split("|")
            aliases = tuple(
                str(alias).strip() for alias in raw_aliases if str(alias).strip()
            )
        except (KeyError, TypeError, ValueError, json.JSONDecodeError) as exc:
            raise ValueError(f"{path}:{line_number}: invalid terminology row") from exc
        if code and label:
            yield TerminologyEntry(code=code, label=label, aliases=aliases)


def _iter_delimited(path: Path) -> Iterable[TerminologyEntry]:
    sample = path.read_text(encoding="utf-8")[:8192]
    delimiter = "\t" if path.suffix.lower() == ".tsv" else ","
    try:
        delimiter = csv.Sniffer().sniff(sample, delimiters="\t,").delimiter
    except csv.Error:
        pass

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter=delimiter)
        fieldnames = {str(name).strip().casefold(): name for name in (reader.fieldnames or [])}
        if "code" not in fieldnames or "label" not in fieldnames:
            raise ValueError(
                f"{path}: expected a header with at least 'code' and 'label' columns"
            )
        for line_number, row in enumerate(reader, start=2):
            code = str(row[fieldnames["code"]] or "").strip()
            label = str(row[fieldnames["label"]] or "").strip()
            aliases_value = (
                str(row.get(fieldnames.get("aliases", ""), "") or "").strip()
                if "aliases" in fieldnames
                else ""
            )
            aliases = tuple(
                alias.strip() for alias in aliases_value.split("|") if alias.strip()
            )
            if not code or not label:
                raise ValueError(f"{path}:{line_number}: code and label are required")
            yield TerminologyEntry(code=code, label=label, aliases=aliases)


def load_terminology(path: Path) -> list[TerminologyEntry]:
    if not path.is_file():
        raise FileNotFoundError(f"Terminology file does not exist: {path}")
    iterator = _iter_jsonl(path) if path.suffix.lower() == ".jsonl" else _iter_delimited(path)
    entries: list[TerminologyEntry] = []
    seen: set[str] = set()
    for entry in iterator:
        if entry.code in seen:
            continue
        seen.add(entry.code)
        entries.append(entry)
    if not entries:
        raise ValueError(f"Terminology file is empty: {path}")
    return entries


class TerminologyIndex:
    """Fast lexical retrieval with an optional local E5 semantic reranker."""

    def __init__(
        self,
        path: Path,
        *,
        embedding_model: str | None = None,
        embedding_device: str = "cpu",
        embedding_cache_dir: Path | None = None,
        embedding_batch_size: int = 128,
    ) -> None:
        self.path = path
        self.entries = load_terminology(path)
        self._normalized_names: list[tuple[str, ...]] = []
        self._tokens: list[set[str]] = []
        self._ngrams: list[set[str]] = []
        self._exact: dict[str, set[int]] = {}
        self._token_postings: dict[str, set[int]] = {}
        self._ngram_postings: dict[str, set[int]] = {}

        for index, entry in enumerate(self.entries):
            names = tuple(
                dict.fromkeys(
                    normalized
                    for normalized in (
                        normalize_term(entry.label),
                        *(normalize_term(alias) for alias in entry.aliases),
                    )
                    if normalized
                )
            )
            token_set = set().union(*(set(name.split()) for name in names))
            ngram_set = set().union(*(_character_ngrams(name) for name in names))
            self._normalized_names.append(names)
            self._tokens.append(token_set)
            self._ngrams.append(ngram_set)
            for name in names:
                self._exact.setdefault(name, set()).add(index)
            for token in token_set:
                if len(token) >= 2:
                    self._token_postings.setdefault(token, set()).add(index)
            for ngram in ngram_set:
                self._ngram_postings.setdefault(ngram, set()).add(index)

        self._embedder = None
        self._embeddings = None
        if embedding_model:
            self._prepare_embeddings(
                embedding_model=embedding_model,
                device=embedding_device,
                cache_dir=embedding_cache_dir,
                batch_size=embedding_batch_size,
            )

    def _prepare_embeddings(
        self,
        *,
        embedding_model: str,
        device: str,
        cache_dir: Path | None,
        batch_size: int,
    ) -> None:
        try:
            import numpy as np
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise RuntimeError(
                "Semantic retrieval requires the local extra: "
                "python -m pip install -e '.[local]'"
            ) from exc

        embedder_key = (embedding_model, device)
        if embedder_key not in _EMBEDDER_CACHE:
            _EMBEDDER_CACHE[embedder_key] = SentenceTransformer(
                embedding_model,
                device=device,
                local_files_only=True,
            )
        self._embedder = _EMBEDDER_CACHE[embedder_key]
        fingerprint_value = (
            f"{self.path.resolve()}:{self.path.stat().st_size}:"
            f"{self.path.stat().st_mtime_ns}:{embedding_model}"
        )
        fingerprint = hashlib.sha256(fingerprint_value.encode("utf-8")).hexdigest()[:20]
        cache_path = None
        if cache_dir is not None:
            cache_dir.mkdir(parents=True, exist_ok=True)
            cache_path = cache_dir / f"{self.path.stem}.{fingerprint}.embeddings.npy"

        if cache_path is not None and cache_path.exists():
            embeddings = np.load(cache_path, mmap_mode="r")
            if embeddings.shape[0] != len(self.entries):
                raise RuntimeError(f"Stale semantic index: {cache_path}")
            self._embeddings = embeddings
            return

        passages = [f"passage: {entry.search_text}" for entry in self.entries]
        embeddings = self._embedder.encode(
            passages,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )
        embeddings = embeddings.astype("float32")
        if cache_path is not None:
            np.save(cache_path, embeddings)
            self._embeddings = np.load(cache_path, mmap_mode="r")
        else:
            self._embeddings = embeddings

    @staticmethod
    def _jaccard(left: set[str], right: set[str]) -> float:
        union = left | right
        return len(left & right) / len(union) if union else 0.0

    def _lexical_score(
        self,
        index: int,
        query: str,
        query_tokens: set[str],
        query_ngrams: set[str],
    ) -> tuple[float, bool]:
        names = self._normalized_names[index]
        exact = query in names
        token_score = self._jaccard(query_tokens, self._tokens[index])
        ngram_score = self._jaccard(query_ngrams, self._ngrams[index])
        contains = any(query in name or name in query for name in names)
        score = (
            (1.0 if exact else 0.0)
            + (0.18 if contains else 0.0)
            + 0.55 * token_score
            + 0.27 * ngram_score
        )
        return min(score, 1.0), exact

    def retrieve(
        self,
        mention: str,
        *,
        context: str = "",
        top_k: int = 20,
        lexical_pool_size: int = 160,
    ) -> list[RetrievedCandidate]:
        query = normalize_term(mention)
        if not query:
            return []
        query_tokens = set(query.split())
        query_ngrams = _character_ngrams(query)

        pool: set[int] = set(self._exact.get(query, set()))
        for token in query_tokens:
            pool.update(self._token_postings.get(token, set()))
        for ngram in query_ngrams:
            pool.update(self._ngram_postings.get(ngram, set()))

        lexical_by_index = {
            index: self._lexical_score(
                index,
                query,
                query_tokens,
                query_ngrams,
            )
            for index in pool
        }

        semantic_scores: dict[int, float] = {}
        if self._embedder is not None and self._embeddings is not None:
            import numpy as np

            semantic_query = normalize_term(f"{mention} {context[:240]}")
            query_embedding = self._embedder.encode(
                [f"query: {semantic_query}"],
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )[0]
            # Add a global semantic pool. This is necessary for Vietnamese
            # mentions whose official ICD/RxNorm labels are only in English.
            all_scores = np.asarray(self._embeddings) @ query_embedding
            semantic_pool_size = min(lexical_pool_size, len(self.entries))
            if semantic_pool_size:
                semantic_indices = np.argpartition(
                    all_scores,
                    -semantic_pool_size,
                )[-semantic_pool_size:]
                pool.update(int(index) for index in semantic_indices)
            semantic_scores = {
                int(index): float((all_scores[index] + 1.0) / 2.0)
                for index in pool
            }

        if not pool:
            return []

        lexical = [
            (*lexical_by_index.get(index, (0.0, False)), index)
            for index in pool
        ]
        lexical.sort(key=lambda item: (-item[0], not item[1], self.entries[item[2]].code))
        lexical = lexical[:lexical_pool_size]

        if semantic_scores:
            # Do not discard semantic-only hits just because their lexical
            # score is zero; rerank the union by the final combined score.
            lexical = [
                (*lexical_by_index.get(index, (0.0, False)), index)
                for index in pool
            ]

        ranked: list[RetrievedCandidate] = []
        for lexical_score, exact, index in lexical:
            semantic_score = semantic_scores.get(index)
            combined = (
                lexical_score
                if semantic_score is None
                else 0.52 * lexical_score + 0.48 * semantic_score
            )
            if not math.isfinite(combined):
                continue
            entry = self.entries[index]
            ranked.append(
                RetrievedCandidate(
                    code=entry.code,
                    label=entry.label,
                    score=combined,
                    exact=exact,
                )
            )
        ranked.sort(key=lambda item: (-item.score, not item.exact, item.code))
        return ranked[:top_k]

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/gliner_ner.py
"""Zero-shot span detection with GLiNER, replacing free-form LLM generation.

Why this exists
---------------
Submission 01 asked Qwen3-8B to *write out* every mention as JSON, then aligned
the generated strings back onto the raw text. That design loses on two fronts the
host metric punishes hardest:

* every mention the model paraphrases is dropped by alignment, so recall is
  capped by the model's copying fidelity;
* generated mentions carry no score, so there is no dial between precision and
  recall — and the scorer counts each spurious concept **twice** in every
  denominator.

GLiNER returns character offsets with a calibrated score per span, which gives
both an exact `text[start:end]` guarantee and a per-type threshold to tune.

Long notes are chunked on line boundaries as verbatim slices, so a chunk offset
plus the local offset is always the true global offset.
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass

from .models import EntityType

LOGGER = logging.getLogger(__name__)

# Descriptive English label strings generalise better than the Vietnamese type
# names: GLiNER was trained on English label prompts.
DEFAULT_LABEL_MAP: dict[str, EntityType] = {
    "symptom": EntityType.SYMPTOM,
    "disease or diagnosis": EntityType.DIAGNOSIS,
    "medication or drug": EntityType.MEDICATION,
    "medical test or lab name": EntityType.TEST_NAME,
    "test result or measurement value": EntityType.TEST_RESULT,
}

# Tuned on the reference solution's dev split; the host double-penalty on
# spurious spans makes precision the right side to err on.
DEFAULT_THRESHOLDS: dict[EntityType, float] = {
    EntityType.SYMPTOM: 0.20,
    EntityType.DIAGNOSIS: 0.25,
    EntityType.MEDICATION: 0.30,
    EntityType.TEST_NAME: 0.15,
    EntityType.TEST_RESULT: 0.35,
}

# A trailing token that belongs inside a THUỐC span. The Vòng 1 example shows
# drug spans carrying strength, route and frequency ("amlodipine 10 mg po daily"),
# but GLiNER usually stops at the drug name.
_DOSE_TOKEN = re.compile(
    r"^(\d+([.,\-/]\d+)*%?"
    r"|\d+\s*(mg|mcg|ug|g|ml|iu|meq|mmol)\b"
    r"|mg|mcg|µg|ug|g|ml|iu|x|%"
    r"|po|iv|im|sc|sl|pr|tab"
    r"|(q\d+h|qd|qid|qod|bid|tid|qhs|qam|qpm|prn|daily)(:prn)?"
    r")$",
    re.IGNORECASE,
)
# Vietnamese indication markers that terminate a drug span.
_INDICATION_MARKERS = ("điều trị", "cho", "để", "khi", "nếu")

# A generic lead-in that adds no clinical content ("dấu hiệu điển hình" -> "điển
# hình" is wrong, but "dấu hiệu vàng da kéo dài" -> "vàng da kéo dài" is right).
_GENERIC_PREFIX = re.compile(
    r"^(?:dấu hiệu|biểu hiện|tình trạng|hội chứng)\b[\s:;,.-]*", re.IGNORECASE
)
_WORD_TOKEN = re.compile(r"[0-9A-Za-zÀ-ỹĐđ]+")


def trim_generic_prefix(text: str, span: "ScoredSpan") -> "ScoredSpan":
    """Drop a generic lead-in when a complete concept (>= 2 words) remains.

    This is the one boundary lever the reference solution kept after ablation;
    the others (symptom-edge trimming, drug re-cleaning) measured worse. A
    one-word remainder is left alone because it is unstable.
    """
    match = _GENERIC_PREFIX.match(text[span.start : span.end])
    if match is None:
        return span
    new_start = span.start + match.end()
    if len(_WORD_TOKEN.findall(text[new_start : span.end])) < 2:
        return span
    return ScoredSpan(new_start, span.end, span.type, span.score)


@dataclass(frozen=True)
class ScoredSpan:
    start: int
    end: int
    type: EntityType
    score: float


def iter_chunks(text: str, max_chunk_chars: int) -> list[tuple[int, str]]:
    """Split into verbatim slices on line boundaries, keeping global offsets."""
    chunks: list[tuple[int, str]] = []
    position = 0
    length = len(text)
    while position < length:
        end = min(position + max_chunk_chars, length)
        if end < length:
            newline = text.rfind("\n", position, end)
            if newline > position:
                end = newline + 1
        chunks.append((position, text[position:end]))
        position = end
    return chunks or [(0, "")]


def extend_medication_span(text: str, start: int, end: int) -> int:
    """Absorb trailing dosage/route/frequency tokens into a medication span."""
    line_end = text.find("\n", end)
    if line_end == -1:
        line_end = len(text)
    remainder = text[end:line_end]
    lowered = remainder.lower()
    for marker in _INDICATION_MARKERS:
        position = lowered.find(marker)
        if position != -1:
            remainder = remainder[:position]
            break
    new_end = end
    for match in re.finditer(r"\S+", remainder):
        if _DOSE_TOKEN.match(match.group(0).strip(".,;:")):
            new_end = end + match.end()
        else:
            break
    return new_end


def resolve_overlaps(spans: list[ScoredSpan]) -> list[ScoredSpan]:
    """Keep non-overlapping spans, preferring higher score then tighter bounds."""
    ordered = sorted(spans, key=lambda span: (-span.score, span.start - span.end))
    kept: list[ScoredSpan] = []
    for span in ordered:
        if any(not (span.end <= other.start or span.start >= other.end) for other in kept):
            continue
        kept.append(span)
    kept.sort(key=lambda span: (span.start, span.end))
    return kept


class GlinerSpanExtractor:
    """Per-type-thresholded GLiNER span extraction over raw clinical text."""

    def __init__(
        self,
        model_path: str = "urchade/gliner_multi-v2.1",
        *,
        label_map: dict[str, EntityType] | None = None,
        thresholds: dict[EntityType, float] | None = None,
        raw_floor: float = 0.02,
        max_chunk_chars: int = 800,
        device: str = "cpu",
    ) -> None:
        try:
            from gliner import GLiNER
        except ImportError as error:  # pragma: no cover - dependency guard
            raise RuntimeError(
                "GLiNER backend needs the gliner package: python -m pip install gliner"
            ) from error

        self.label_map = label_map or DEFAULT_LABEL_MAP
        self.labels = list(self.label_map)
        self.thresholds = thresholds or DEFAULT_THRESHOLDS
        self.raw_floor = raw_floor
        self.max_chunk_chars = max_chunk_chars
        LOGGER.info("loading GLiNER %s on %s", model_path, device)
        self.model = GLiNER.from_pretrained(model_path)
        try:
            self.model = self.model.to(device)
        except Exception:  # pragma: no cover - some versions manage device internally
            LOGGER.debug("GLiNER manages its own device placement")
        self.model.eval()
        self.parameters = sum(
            parameter.numel() for parameter in self.model.parameters()
        )

    def scored_spans(self, text: str) -> list[ScoredSpan]:
        """All spans above ``raw_floor``, before per-type gating."""
        spans: list[ScoredSpan] = []
        for chunk_start, chunk in iter_chunks(text, self.max_chunk_chars):
            if not chunk.strip():
                continue
            predictions = self.model.predict_entities(
                chunk, self.labels, threshold=self.raw_floor
            )
            for prediction in predictions:
                entity_type = self.label_map.get(prediction["label"])
                if entity_type is None:
                    continue
                start = chunk_start + prediction["start"]
                end = chunk_start + prediction["end"]
                while start < end and text[start].isspace():
                    start += 1
                while end > start and text[end - 1].isspace():
                    end -= 1
                if entity_type is EntityType.MEDICATION:
                    end = extend_medication_span(text, start, end)
                if start < end:
                    spans.append(
                        ScoredSpan(start, end, entity_type, float(prediction.get("score", 1.0)))
                    )
        return spans

    def gate(self, spans: list[ScoredSpan]) -> list[ScoredSpan]:
        """Apply per-type thresholds, then resolve overlaps."""
        return resolve_overlaps(
            [span for span in spans if span.score >= self.thresholds.get(span.type, 0.5)]
        )

    def extract(self, text: str) -> list[ScoredSpan]:
        return self.gate(self.scored_spans(text))

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/exact_link.py
"""Precision-first candidate linking: one code, only on a unique exact alias.

Rationale from the metric, not from taste. In the host scorer a diagnosis or
medication concept contributes with weight ``w = len(gold_candidates) + 1``:

* when the gold list is **empty**, predicting empty scores Jaccard 1.0 and
  predicting any code scores 0.0;
* when the concept is **spurious** it lands in the denominator as
  ``2 * (len(predicted_candidates) + 1)`` — so emitting three codes on a wrong
  span costs 8 denominator units instead of 2.

Both effects push the same way: emit a code only when it is nearly certain. The
reference solution measured this directly — removing *every* candidate moved
their candidate Jaccard by only 0.0036, while their retrieve-and-rerank variant
(SapBERT + Qwen listwise) scored materially worse than this policy because it
always picked something.

So retrieval breadth and emission breadth are separated: look up widely, emit
only on a unique exact alias match.
"""
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

from .models import EntityType
from .terminology import load_terminology

# Route / frequency / dose noise that is not part of a canonical drug name.
_DRUG_NOISE = re.compile(
    r"\b(po|iv|im|sc|sl|pr|tid|bid|qd|qid|qhs|qam|qpm|prn|q\d+h(:prn)?|daily|twice|once|"
    r"uống|tiêm|truyền|lần|ngày|viên|tab|caps?)\b",
    re.IGNORECASE,
)
_DECIMAL_COMMA = re.compile(r"(\d),(\d)")
_UNIT_SPACING = re.compile(r"\s*(mg|mcg|µg|ug|g|ml|iu|meq|mmol)\b", re.IGNORECASE)

_BARE_ICD_CATEGORY = re.compile(r"^[A-Z]\d\d$")


def normalize_surface(value: str) -> str:
    """Lowercase and collapse whitespace for lookup only; never touches offsets."""
    return " ".join(str(value).split()).strip(" ,;:.").lower()


def strip_diacritics(value: str) -> str:
    decomposed = unicodedata.normalize("NFD", value)
    stripped = "".join(
        character for character in decomposed if unicodedata.category(character) != "Mn"
    )
    return unicodedata.normalize("NFC", stripped).replace("đ", "d")


def clean_mention(mention: str, entity_type: EntityType) -> str:
    text = " ".join(mention.split())
    if entity_type is EntityType.MEDICATION:
        text = _DRUG_NOISE.sub(" ", text)
        text = _DECIMAL_COMMA.sub(r"\1.\2", text)
        text = _UNIT_SPACING.sub(r" \1", text)
        text = " ".join(text.split())
    return text.strip()


class ExactAliasIndex:
    """Alias -> codes map with a diacritic-insensitive fallback."""

    def __init__(self, entries) -> None:
        self.codes: set[str] = set()
        self._exact: dict[str, set[str]] = {}
        self._plain: dict[str, set[str]] = {}
        for entry in entries:
            self.codes.add(entry.code)
            for surface in (entry.label, *entry.aliases):
                key = normalize_surface(surface)
                if not key:
                    continue
                self._exact.setdefault(key, set()).add(entry.code)
                self._plain.setdefault(strip_diacritics(key), set()).add(entry.code)

    @classmethod
    def from_path(cls, path: Path) -> "ExactAliasIndex":
        return cls(load_terminology(path))

    def lookup(self, mention: str) -> set[str]:
        key = normalize_surface(mention)
        if not key:
            return set()
        hit = self._exact.get(key)
        if hit:
            return hit
        return self._plain.get(strip_diacritics(key), set())


class PrecisionFirstLinker:
    """Emit at most one code, and only when exactly one alias matches."""

    def __init__(
        self,
        icd_index: ExactAliasIndex | None = None,
        rxnorm_index: ExactAliasIndex | None = None,
        *,
        max_candidates: int = 1,
        leaf_remap: bool = True,
    ) -> None:
        self.icd_index = icd_index
        self.rxnorm_index = rxnorm_index
        self.max_candidates = max_candidates
        self.leaf_remap = leaf_remap
        self._cache: dict[tuple[str, str], list[str]] = {}

    def _index_for(self, entity_type: EntityType) -> ExactAliasIndex | None:
        if entity_type is EntityType.DIAGNOSIS:
            return self.icd_index
        if entity_type is EntityType.MEDICATION:
            return self.rxnorm_index
        return None

    def _leafify(self, code: str) -> str:
        """Remap a bare 3-character ICD category to its ``.9`` leaf when it exists.

        A bare-category mention carries no complication or severity detail, and
        the target is almost always a leaf, so ``.9`` (unspecified) is the
        conventional resolution. This can only turn a miss into a hit.
        """
        if not self.leaf_remap or self.icd_index is None:
            return code
        if _BARE_ICD_CATEGORY.match(code) and f"{code}.9" in self.icd_index.codes:
            return f"{code}.9"
        return code

    def link(self, mention: str, entity_type: EntityType) -> list[str]:
        index = self._index_for(entity_type)
        if index is None:
            return []
        cleaned = clean_mention(mention, entity_type)
        if not cleaned:
            return []
        key = (entity_type.value, cleaned)
        cached = self._cache.get(key)
        if cached is not None:
            return list(cached)

        codes = index.lookup(cleaned)
        # Precision-first: ambiguity is a reason to stay silent, not to guess.
        if len(codes) != 1:
            result: list[str] = []
        else:
            code = next(iter(codes))
            if entity_type is EntityType.DIAGNOSIS:
                code = self._leafify(code)
            result = [code][: self.max_candidates]
        self._cache[key] = result
        return list(result)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/selector.py
"""Qwen teachers that re-type and extend GLiNER spans using next-token logits.

Two roles, both decided by a *single forward pass* over a fixed label set — never
by generation. That matters: a generated answer can be anything, while a logit
over the digits 0-5 is a closed decision that cannot corrupt the schema.

1. **Corrector** — for every baseline ``TRIỆU_CHỨNG`` span, read five-way logits
   and re-type it to ``CHẨN_ĐOÁN`` when the teacher disagrees. This is GLiNER's
   single largest type error: it reads chronic diseases as symptoms. Our
   CPU-only run produced 529 diagnoses against the reference solution's 668, and
   this step is what closes that gap.

2. **Additions** — for low-scoring raw spans that do not overlap a baseline span,
   read six-way logits (0 = not a concept) in *both* teachers and add the span
   only when they agree on the same type, that type carries no candidates, and
   both clear a margin over ``NONE``.

Additions are restricted to ``TRIỆU_CHỨNG`` / ``TÊN_XÉT_NGHIỆM`` /
``KẾT_QUẢ_XÉT_NGHIỆM`` on purpose. A spurious diagnosis or drug is charged to the
candidate denominator as well (``2 * (n_codes + 1)``), so recall bought in those
two types is paid for twice.

Consensus is required for additions but not for correction: correction moves a
span that already exists, while an addition creates one, and only the second can
manufacture a spurious concept.
"""
from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Iterable, Sequence

from .gliner_ner import ScoredSpan
from .models import EntityType

LOGGER = logging.getLogger(__name__)

# Digit order is fixed: the prompts below number the types 1-5 in this sequence.
ORDERED_TYPES: tuple[EntityType, ...] = (
    EntityType.SYMPTOM,
    EntityType.DIAGNOSIS,
    EntityType.MEDICATION,
    EntityType.TEST_NAME,
    EntityType.TEST_RESULT,
)

# Types safe to add: none of them carry candidates.
DEFAULT_ADDITION_TYPES = frozenset(
    {EntityType.SYMPTOM, EntityType.TEST_NAME, EntityType.TEST_RESULT}
)

FIVE_WAY_SYSTEM = (
    "Bạn là chuyên gia y lâm sàng Việt Nam. Phân loại ý niệm y khoa vào ĐÚNG MỘT loại:\n"
    "1 = TRIỆU_CHỨNG (triệu chứng: đau đầu, sốt, phù, mệt mỏi)\n"
    "2 = CHẨN_ĐOÁN (chẩn đoán bệnh: đái tháo đường, tăng huyết áp, viêm phổi, ung thư)\n"
    "3 = THUỐC (paracetamol, ceftriaxone, amlodipine)\n"
    "4 = TÊN_XÉT_NGHIỆM (công thức máu, MRI, nội soi, X-quang)\n"
    "5 = KẾT_QUẢ_XÉT_NGHIỆM (giá trị số: 120 mg/dL, HbA1c 7.2%, GCS 15)\n"
    "Quy tắc: 'tăng huyết áp/đái tháo đường/xơ gan' = 2; 'đau/phù/sốt/mệt' = 1."
)

SIX_WAY_SYSTEM = (
    "Bạn là chuyên gia y lâm sàng Việt Nam. Phân loại ý niệm y khoa:\n"
    "0 = KHÔNG PHẢI ý niệm y khoa hợp lệ (tiêu đề, từ chung chung, số/liều rời, hành chính)\n"
    "1 = TRIỆU_CHỨNG (triệu chứng: đau đầu, sốt, phù)\n"
    "2 = CHẨN_ĐOÁN (bệnh: đái tháo đường, tăng huyết áp, viêm phổi)\n"
    "3 = THUỐC (paracetamol, ceftriaxone)\n"
    "4 = TÊN_XÉT_NGHIỆM (công thức máu, MRI, nội soi)\n"
    "5 = KẾT_QUẢ_XÉT_NGHIỆM (giá trị số: 120 mg/dL, GCS 15)\n"
    "Quy tắc: 'tăng huyết áp/đái tháo đường'=2; 'đau/phù/sốt'=1; tiêu đề=0."
)


def line_context(text: str, start: int, end: int, line_limit: int = 300) -> str:
    """The mention's line, prefixed by the nearest non-empty line as a section hint."""
    line_start = text.rfind("\n", 0, start) + 1
    line_end = text.find("\n", end)
    if line_end == -1:
        line_end = len(text)
    line = text[line_start:line_end][:line_limit]

    previous = ""
    cursor = line_start - 1
    while cursor > 0:
        candidate_start = text.rfind("\n", 0, cursor) + 1
        candidate_end = text.find("\n", candidate_start)
        if candidate_end == -1:
            candidate_end = len(text)
        candidate = text[candidate_start:candidate_end].strip()
        if candidate:
            previous = candidate[:80]
            break
        cursor = candidate_start - 1
    return f"[mục: {previous}] {line}" if previous else line


def single_token_ids(tokenizer, surfaces: Iterable[str]) -> list[int]:
    """Token ids for surfaces that encode to exactly one token."""
    ids = []
    for surface in surfaces:
        encoded = tokenizer.encode(surface, add_special_tokens=False)
        if len(encoded) == 1:
            ids.append(encoded[0])
    return ids


def digit_token_groups(tokenizer, digits: Iterable[int]) -> list[list[int]]:
    """One id group per digit, covering the bare and space-prefixed forms."""
    return [single_token_ids(tokenizer, (str(d), f" {d}")) for d in digits]


@dataclass
class TeacherConfig:
    model_path: str
    device: str = "cuda"
    quantization: str = "4bit"
    dtype: str = "bfloat16"


class Teacher:
    """A causal LM used only to score a fixed set of next tokens."""

    def __init__(self, config: TeacherConfig) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        torch_dtype = {
            "bfloat16": torch.bfloat16,
            "float16": torch.float16,
            "float32": torch.float32,
        }.get(config.dtype, torch.bfloat16)

        self.tokenizer = AutoTokenizer.from_pretrained(config.model_path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        if config.quantization == "none":
            self.model = AutoModelForCausalLM.from_pretrained(
                config.model_path, torch_dtype=torch_dtype
            ).to(config.device)
        else:
            from transformers import BitsAndBytesConfig

            if config.quantization == "4bit":
                bnb = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch_dtype,
                    bnb_4bit_use_double_quant=True,
                )
            elif config.quantization == "8bit":
                bnb = BitsAndBytesConfig(load_in_8bit=True)
            else:
                raise ValueError(f"Unknown quantization mode: {config.quantization!r}")
            self.model = AutoModelForCausalLM.from_pretrained(
                config.model_path,
                quantization_config=bnb,
                device_map={"": config.device},
            )
        self.model.eval()
        self.parameters = sum(p.numel() for p in self.model.parameters())
        self.entity_digits = digit_token_groups(self.tokenizer, range(1, 6))
        self.none_digits = single_token_ids(self.tokenizer, ("0", " 0"))
        LOGGER.info(
            "loaded teacher %s (%.3fB params) on %s",
            config.model_path,
            self.parameters / 1e9,
            config.device,
        )

    def chat_prompt(self, system: str, user: str) -> str:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        try:
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:  # older templates have no enable_thinking argument
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

    def group_scores(
        self,
        prompts: Sequence[str],
        groups: Sequence[Sequence[int]],
        batch_size: int,
        max_length: int,
    ) -> list[list[float]]:
        """Log-sum-exp of each token group, from one next-token forward per batch."""
        import torch

        device = next(self.model.parameters()).device
        results: list[list[float]] = []
        for offset in range(0, len(prompts), batch_size):
            batch = list(prompts[offset : offset + batch_size])
            encoded = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length + 8,
            ).to(device)
            with torch.inference_mode():
                try:
                    logits = self.model(**encoded, logits_to_keep=1).logits[:, -1, :]
                except TypeError:  # transformers without logits_to_keep
                    logits = self.model(**encoded).logits[:, -1, :]
            for row in logits:
                results.append(
                    [
                        torch.logsumexp(row[list(ids)], -1).item() if ids else -1e9
                        for ids in groups
                    ]
                )
        return results


class SpanSelector:
    """Type correction, plus consensus additions when a second teacher is present."""

    def __init__(
        self,
        primary: Teacher,
        secondary: Teacher | None = None,
        *,
        batch_size: int = 48,
        max_length: int = 384,
        addition_margin: float = 1.0,
        addition_types: frozenset[EntityType] = DEFAULT_ADDITION_TYPES,
    ) -> None:
        self.primary = primary
        self.secondary = secondary
        self.batch_size = batch_size
        self.max_length = max_length
        self.addition_margin = addition_margin
        self.addition_types = addition_types

    @property
    def total_parameters(self) -> int:
        total = self.primary.parameters
        if self.secondary is not None:
            total += self.secondary.parameters
        return total

    def _five_way(self, teacher: Teacher, prompts: Sequence[str]) -> list[EntityType]:
        scores = teacher.group_scores(
            prompts, teacher.entity_digits, self.batch_size, self.max_length
        )
        return [ORDERED_TYPES[max(range(5), key=row.__getitem__)] for row in scores]

    def _six_way(
        self, teacher: Teacher, prompts: Sequence[str]
    ) -> list[tuple[EntityType, float]]:
        groups = list(teacher.entity_digits) + [teacher.none_digits]
        scores = teacher.group_scores(prompts, groups, self.batch_size, self.max_length)
        output = []
        for row in scores:
            entity_scores, none_score = row[:5], row[5]
            best = max(range(5), key=entity_scores.__getitem__)
            output.append((ORDERED_TYPES[best], entity_scores[best] - none_score))
        return output

    def correct_types(self, text: str, spans: list[ScoredSpan]) -> list[ScoredSpan]:
        """Re-type TRIỆU_CHỨNG spans the primary teacher reads as CHẨN_ĐOÁN."""
        indices = [i for i, span in enumerate(spans) if span.type is EntityType.SYMPTOM]
        if not indices:
            return list(spans)
        prompts = [
            self.primary.chat_prompt(
                FIVE_WAY_SYSTEM,
                f"Ngữ cảnh: {line_context(text, spans[i].start, spans[i].end)}\n"
                f'Ý niệm: "{text[spans[i].start : spans[i].end]}"\n'
                "Trả lời CHỈ MỘT chữ số 1-5.",
            )
            for i in indices
        ]
        predictions = self._five_way(self.primary, prompts)
        corrected = list(spans)
        for position, prediction in zip(indices, predictions):
            if prediction is EntityType.DIAGNOSIS:
                span = spans[position]
                corrected[position] = ScoredSpan(
                    span.start, span.end, EntityType.DIAGNOSIS, span.score
                )
        return corrected

    def propose_additions(
        self,
        text: str,
        baseline: list[ScoredSpan],
        raw: list[ScoredSpan],
    ) -> list[ScoredSpan]:
        """Add non-overlapping raw spans both teachers agree on."""
        if self.secondary is None:
            return []
        occupied = [(span.start, span.end) for span in baseline]

        def overlaps(start: int, end: int) -> bool:
            return any(not (end <= s or start >= e) for s, e in occupied)

        candidates: list[tuple[int, int]] = []
        seen: set[tuple[int, int]] = set()
        for span in raw:
            key = (span.start, span.end)
            if key in seen or overlaps(*key):
                continue
            if len(text[span.start : span.end].strip()) < 2:
                continue
            seen.add(key)
            candidates.append(key)
        if not candidates:
            return []

        prompts = [
            self.primary.chat_prompt(
                SIX_WAY_SYSTEM,
                f"Ngữ cảnh: {line_context(text, start, end)}\n"
                f'Ý niệm: "{text[start:end]}"\n'
                "Trả lời CHỈ MỘT chữ số 0-5.",
            )
            for start, end in candidates
        ]
        primary_votes = self._six_way(self.primary, prompts)
        secondary_votes = self._six_way(self.secondary, prompts)

        additions: list[ScoredSpan] = []
        for (start, end), (p_type, p_margin), (s_type, s_margin) in zip(
            candidates, primary_votes, secondary_votes
        ):
            if p_type is not s_type or p_type not in self.addition_types:
                continue
            if p_margin < self.addition_margin or s_margin < self.addition_margin:
                continue
            additions.append(ScoredSpan(start, end, p_type, 0.0))
        return additions

    def select(
        self,
        text: str,
        baseline: list[ScoredSpan],
        raw: list[ScoredSpan],
    ) -> list[ScoredSpan]:
        corrected = self.correct_types(text, baseline)
        corrected.extend(self.propose_additions(text, corrected, raw))
        corrected.sort(key=lambda span: (span.start, span.end))
        return corrected

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/pipeline_v2.py
"""Precision-first pipeline: GLiNER spans + exact-alias linking + empty assertions.

    raw text
    -> GLiNER spans (per-type thresholds, verbatim offsets)
    -> overlap resolution
    -> header/junk rejection
    -> unique-exact-alias linking against the Vietnamese ICD-10 (TT06) and RxNorm
    -> schema validation
    -> output.zip

There is no generative step, so the whole run is CPU-only and finishes in about a
minute for 100 records. That matters beyond speed: every design choice below is
one the host metric rewards directly, and none of them depend on a GPU being
available at submission time.

Assertions are always emitted empty, and that is a measured choice rather than a
gap. A wrong assertion forfeits the whole Jaccard of its concept, while an empty
prediction against an empty gold list scores 1.0; on the reference solution's
split every negation / family / history rule they tried over-fired, and
`isNegated` separated at AUC 0.497 — chance. Submission 01 emitted 266 assertion
labels and scored 20.19 on the component; the reference emitted none and scored
35.27. Restoring assertions is worth revisiting only against labelled validation
data, which is why there is no flag to half-enable it here.
"""
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass
from pathlib import Path

from .exact_link import ExactAliasIndex, PrecisionFirstLinker, strip_diacritics
from .gliner_ner import (
    DEFAULT_THRESHOLDS,
    GlinerSpanExtractor,
    ScoredSpan,
    resolve_overlaps,
    trim_generic_prefix,
)
from .models import AlignedEntity, EntityType
from .validation import validate_submission_record

# The declared budget is on parameter count, not memory: quantization does not
# change what must be reported. GLiNER 0.289B + two 4B teachers = 8.517B.
PARAMETER_BUDGET = 9_000_000_000

LOGGER = logging.getLogger(__name__)

# Bare field labels and section headings. A span is rejected only when it is
# *exactly* one of these: the same words inside a longer mention ("kết quả xét
# nghiệm glucose cao") are part of a real concept and must survive, so this is an
# equality test rather than a substring blacklist.
HEADER_LABELS = frozenset(
    {
        "thuốc", "tên thuốc",
        "triệu chứng", "các triệu chứng",
        "chẩn đoán", "chẩn đoán sơ bộ", "chẩn đoán ra viện", "icd",
        "xét nghiệm", "tên xét nghiệm",
        "kết quả", "kết quả xét nghiệm",
        "khám lâm sàng", "cận lâm sàng",
        "tiền sử", "tiền sử bệnh", "bệnh sử",
        "điều trị", "tình trạng", "diễn biến",
        "lý do", "lý do vào viện", "hỏi bệnh",
        "bệnh nhân",
    }
)
_NUMBERED_HEADING = re.compile(r"^\d+\s*[.)]\s*$")


@dataclass(frozen=True)
class PipelineV2Config:
    input_dir: Path
    output_dir: Path
    model_path: str = "urchade/gliner_multi-v2.1"
    device: str = "cpu"
    icd_kb: Path | None = None
    rxnorm_kb: Path | None = None
    thresholds: dict[EntityType, float] | None = None
    raw_floor: float = 0.02
    max_chunk_chars: int = 800
    max_candidates: int = 1
    selected_ids: frozenset[str] | None = None
    # Optional Qwen teachers. Without them the run is CPU-only; with them the
    # TRIỆU_CHỨNG -> CHẨN_ĐOÁN corrector runs, and additions need both.
    primary_teacher: str | None = None
    secondary_teacher: str | None = None
    teacher_device: str = "cuda"
    teacher_quantization: str = "4bit"
    teacher_batch_size: int = 48
    addition_margin: float = 1.0


def is_header_span(text: str) -> bool:
    """True when the mention is a bare heading rather than a clinical concept."""
    stripped = " ".join(text.split())
    if len(stripped) < 2:
        return True
    if _NUMBERED_HEADING.match(stripped):
        return True
    normalized = stripped.rstrip(":;.-").strip().lower()
    if normalized in HEADER_LABELS:
        return True
    return strip_diacritics(normalized) in {
        strip_diacritics(label) for label in HEADER_LABELS
    }


def discover_inputs(input_dir: Path, selected_ids: frozenset[str] | None) -> list[Path]:
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")
    files = [path for path in input_dir.glob("*.txt") if path.stem.isdigit()]
    files.sort(key=lambda path: int(path.stem))
    if selected_ids is not None:
        files = [path for path in files if path.stem in selected_ids]
        missing = selected_ids - {path.stem for path in files}
        if missing:
            raise FileNotFoundError(f"Input IDs not found: {sorted(missing, key=int)}")
    if not files:
        raise FileNotFoundError(f"No numeric .txt files found in {input_dir}")
    return files


def build_linker(config: PipelineV2Config) -> PrecisionFirstLinker:
    icd_index = ExactAliasIndex.from_path(config.icd_kb) if config.icd_kb else None
    rxnorm_index = ExactAliasIndex.from_path(config.rxnorm_kb) if config.rxnorm_kb else None
    if icd_index is None:
        LOGGER.warning("No --icd-kb given; CHẨN_ĐOÁN candidates will all be empty")
    if rxnorm_index is None:
        LOGGER.warning("No --rxnorm-kb given; THUỐC candidates will all be empty")
    return PrecisionFirstLinker(
        icd_index=icd_index,
        rxnorm_index=rxnorm_index,
        max_candidates=config.max_candidates,
    )


def build_entities(
    raw_text: str,
    spans: list[ScoredSpan],
    linker: PrecisionFirstLinker,
) -> list[AlignedEntity]:
    entities: list[AlignedEntity] = []
    for span in spans:
        mention = raw_text[span.start : span.end]
        if is_header_span(mention):
            continue
        entities.append(
            AlignedEntity(
                text=mention,
                type=span.type,
                assertions=[],
                position=(span.start, span.end),
                candidates=linker.link(mention, span.type),
            )
        )
    return entities


def _atomic_write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def build_selector(config: PipelineV2Config, gliner_parameters: int):
    """Load the Qwen teachers, refusing to exceed the declared parameter budget."""
    if config.primary_teacher is None:
        return None
    from .selector import SpanSelector, Teacher, TeacherConfig

    def load(path: str):
        return Teacher(
            TeacherConfig(
                model_path=path,
                device=config.teacher_device,
                quantization=config.teacher_quantization,
            )
        )

    primary = load(config.primary_teacher)
    secondary = load(config.secondary_teacher) if config.secondary_teacher else None
    selector = SpanSelector(
        primary,
        secondary,
        batch_size=config.teacher_batch_size,
        addition_margin=config.addition_margin,
    )
    total = gliner_parameters + selector.total_parameters
    if total > PARAMETER_BUDGET:
        raise RuntimeError(
            f"Declared parameters {total / 1e9:.3f}B exceed the {PARAMETER_BUDGET / 1e9:.0f}B "
            "limit. Drop the secondary teacher or use smaller ones."
        )
    LOGGER.info("declared parameters: %.3fB / %.0fB", total / 1e9, PARAMETER_BUDGET / 1e9)
    if secondary is None:
        LOGGER.info("no secondary teacher: type correction only, no span additions")
    return selector


def run_pipeline_v2(config: PipelineV2Config) -> int:
    inputs = discover_inputs(config.input_dir, config.selected_ids)
    config.output_dir.mkdir(parents=True, exist_ok=True)

    extractor = GlinerSpanExtractor(
        config.model_path,
        thresholds=config.thresholds or DEFAULT_THRESHOLDS,
        raw_floor=config.raw_floor,
        max_chunk_chars=config.max_chunk_chars,
        device=config.device,
    )
    linker = build_linker(config)
    selector = build_selector(config, extractor.parameters)

    total = 0
    for index, input_path in enumerate(inputs, start=1):
        raw_text = input_path.read_text(encoding="utf-8")
        raw_spans = extractor.scored_spans(raw_text)
        spans = extractor.gate(raw_spans)
        if selector is not None:
            spans = selector.select(raw_text, spans, raw_spans)
        spans = resolve_overlaps(
            [trim_generic_prefix(raw_text, span) for span in spans]
        )
        entities = build_entities(raw_text, spans, linker)
        submission = [entity.to_submission_dict() for entity in entities]
        validate_submission_record(raw_text, submission)
        _atomic_write_json(config.output_dir / f"{input_path.stem}.json", submission)
        total += len(submission)
        LOGGER.info(
            "[%d/%d] %s: %d concepts", index, len(inputs), input_path.stem, len(submission)
        )
    return total

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/icd_vn.py
"""Build a Vietnamese ICD-10 terminology table from the official MoH catalog.

Source: *Phụ lục Bảng danh mục mã ICD-10 tiếng Việt*, issued with Thông tư
06/2026/TT-BYT (TT06), published by Bộ Y tế. The catalog ships in
``data/kb/raw/`` and is the reason a Vietnamese note can be linked at all: the
CDC ICD-10-CM descriptions used by the first submission are English only, so a
mention like ``viêm túi mật`` could never match an alias.

Three Vietnamese surfaces are harvested per row:

* ``TÊN BỆNH`` -> ``MÃ BỆNH``                        (leaf, e.g. A00.0)
* ``TÊN NHÓM BỆNH 3 KÝ TỰ`` -> ``MÃ NHÓM BỆNH 3 KÝ TỰ`` (3-char category, A00)
* ``HƯỚNG DẪN MÃ HÓA BỔ SUNG CỦA WHO 2019`` -> ``MÃ BỆNH`` (WHO synonyms)

The third column is not used by the reference solution. It is a mixed field: some
rows are clean synonyms (``Bệnh tả cổ điển``), others are inclusion notes
(``Bao gồm: ...``) or dagger/asterisk cross-references. :func:`iter_who_synonyms`
keeps only the clean synonym surfaces, which adds ~2.9k aliases and lifts the
unique-exact-match rate on our diagnosis mentions from 9.0% to 10.9%.

Output is the ``code / label / aliases`` TSV that :mod:`medical_coder.terminology`
already reads, so the result is a drop-in for ``--icd-kb``.
"""
from __future__ import annotations

import csv
import re
from pathlib import Path

CATEGORY_CODE_COLUMN = "MÃ NHÓM BỆNH 3 KÝ TỰ"
CATEGORY_NAME_COLUMN = "TÊN NHÓM BỆNH 3 KÝ TỰ"
LEAF_CODE_COLUMN = "MÃ BỆNH"
LEAF_NAME_COLUMN = "TÊN BỆNH"
WHO_SYNONYM_COLUMN = "HƯỚNG DẪN MÃ HÓA BỔ SUNG CỦA WHO 2019"

# COVID-19 codes from QĐ 98, absent from the TT06 annex.
COVID_CODES = (
    ("U07.1", "COVID-19, vi rút được xác định"),
    ("U07.2", "COVID-19, vi rút không được xác định"),
)

# Inclusion/exclusion prose in the WHO guidance column, never a usable synonym.
_GUIDANCE_PROSE = re.compile(
    r"(bao gồm|loại trừ|dùng thêm|sử dụng|xem |mã hóa)", re.IGNORECASE
)


def dotted_code(value: str) -> str:
    """Normalize an ICD-10 code to dotted form (``A001`` -> ``A00.1``)."""
    code = str(value).strip().upper().replace(".", "")
    return f"{code[:3]}.{code[3:]}" if len(code) > 3 else code


def clean_name(value: str) -> str:
    return " ".join(str(value).replace("・", " ").split()).strip(" ,;:")


def iter_who_synonyms(value: str):
    """Yield usable Vietnamese synonyms from the WHO guidance column.

    Rejects the whole cell when it carries cross-reference markup (``†``, ``*``,
    parenthesised codes) or reads as inclusion/exclusion prose, because those
    surfaces are not names a clinician would write in a note.
    """
    raw = str(value).strip()
    if not raw or raw.isdigit():
        return
    if "†" in raw or "*" in raw or "(" in raw or _GUIDANCE_PROSE.search(raw):
        return
    for part in re.split(r"[\n;]+", raw):
        candidate = part.strip().strip("+-–— ")
        if not candidate or candidate.endswith(":") or candidate.isdigit():
            return
        if 4 <= len(candidate) <= 90:
            yield candidate


def build_alias_table(xlsx_path: Path) -> dict[str, dict[str, object]]:
    """Return ``{code: {"label": str, "aliases": list[str]}}`` from the catalog."""
    try:
        import pandas as pd
    except ImportError as error:  # pragma: no cover - dependency guard
        raise RuntimeError(
            "Building the Vietnamese ICD KB needs pandas and openpyxl: "
            "python -m pip install pandas openpyxl"
        ) from error

    sheet = pd.read_excel(xlsx_path, sheet_name=0, header=None, dtype=str).fillna("")

    header_row = None
    for index in range(min(15, len(sheet))):
        if any(str(cell).strip().upper() == LEAF_CODE_COLUMN for cell in sheet.iloc[index]):
            header_row = index
            break
    if header_row is None:
        raise ValueError(f"Could not find a '{LEAF_CODE_COLUMN}' header in {xlsx_path}")

    header = [str(cell).strip() for cell in sheet.iloc[header_row]]
    body = sheet.iloc[header_row + 1 :].reset_index(drop=True)
    body.columns = header

    def column(name: str) -> str | None:
        for candidate in header:
            if candidate.strip().upper() == name:
                return candidate
        return None

    leaf_code = column(LEAF_CODE_COLUMN)
    leaf_name = column(LEAF_NAME_COLUMN)
    if leaf_code is None or leaf_name is None:
        raise ValueError(f"Missing {LEAF_CODE_COLUMN}/{LEAF_NAME_COLUMN} in {xlsx_path}")
    category_code = column(CATEGORY_CODE_COLUMN)
    category_name = column(CATEGORY_NAME_COLUMN)
    synonym_column = column(WHO_SYNONYM_COLUMN)

    table: dict[str, dict[str, object]] = {}

    def add(raw_code: str, raw_name: str) -> None:
        code = dotted_code(raw_code)
        if len(code.replace(".", "")) < 3:
            return
        name = clean_name(raw_name)
        if not name or name.lower() == "nan":
            return
        entry = table.setdefault(code, {"label": name, "aliases": []})
        aliases: list[str] = entry["aliases"]  # type: ignore[assignment]
        if name != entry["label"] and name not in aliases:
            aliases.append(name)

    for _, row in body.iterrows():
        add(row[leaf_code], row[leaf_name])
        if category_code is not None and category_name is not None:
            add(row[category_code], row[category_name])
        if synonym_column is not None:
            for synonym in iter_who_synonyms(row[synonym_column]):
                add(row[leaf_code], synonym)

    for code, name in COVID_CODES:
        add(code, name)
    return table


def write_terminology_tsv(table: dict[str, dict[str, object]], destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["code", "label", "aliases"])
        for code in sorted(table):
            entry = table[code]
            writer.writerow([code, entry["label"], "|".join(entry["aliases"])])  # type: ignore[arg-type]
    return len(table)


def build(xlsx_path: Path, destination: Path) -> int:
    return write_terminology_tsv(build_alias_table(xlsx_path), destination)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/rxnorm_kb.py
"""Build an RxNorm terminology table from RxNorm Current Prescribable Content.

The archive (``RxNorm_full_prescribe_<date>.zip``, public, no UMLS licence) is not
redistributable with the repository, so this module builds the TSV from a local
copy — see KAGGLE.md for the download step.

Only names that a clinician would actually write are kept. The term types below
cover the ingredient / brand / clinical-drug surfaces that appear in the Vòng 1
example (``amlodipine 10 mg po daily`` -> 308135, a clinical drug), while
dropping dose-form and pack noise that never matches a mention.
"""
from __future__ import annotations

import csv
import zipfile
from pathlib import Path

# RXNCONSO.RRF column offsets (pipe-delimited, no header).
RXCUI, SAB, TTY, STR, SUPPRESS = 0, 11, 12, 14, 16

# Concept classes a mention may legitimately resolve to. Every gold RxCUI in the
# Vòng 1 example is an SCD (``amlodipine 10 mg po daily`` -> 308135, "amlodipine
# 10 MG Oral Tablet"), plus ingredients and brand names for bare drug words.
#
# SCDC (ingredient + strength, e.g. 329526 "amlodipine 10 MG") is deliberately
# excluded. A dosed mention cleans to exactly an SCDC surface, so indexing it
# turns "no answer" into a *confidently wrong* answer on precisely the mentions
# most likely to carry a gold code. SCDC maps one-to-many onto SCDs by dose form,
# which is unresolvable from the mention — so under a unique-match policy the
# right output is silence.
CONCEPT_TERM_TYPES = frozenset({"SCD", "SBD", "BN", "IN", "PIN", "MIN"})

# Name rows worth indexing once a concept qualifies above. SY/PSN/TMSY are
# alternate surfaces of the same RxCUI (243670 SY "ASA 81 MG Oral Tablet").
NAME_TERM_TYPES = CONCEPT_TERM_TYPES | {"SY", "PSN", "TMSY"}

# Compact surfaces preferred as the display label.
LABEL_TERM_TYPES = frozenset({"IN", "PIN", "BN"})


def _iter_rows(handle):
    for line in handle:
        fields = line.decode("utf-8", "replace").rstrip("\n").split("|")
        if len(fields) <= SUPPRESS:
            continue
        if fields[SAB] != "RXNORM" or fields[SUPPRESS] not in ("N", ""):
            continue
        name = fields[STR].strip()
        if name:
            yield fields[RXCUI].strip(), fields[TTY], name


def build_alias_table(archive: Path) -> dict[str, dict[str, object]]:
    """Index every name of each RxCUI that qualifies as a linkable concept.

    Qualification is decided per *concept*, not per row: an SCDC RxCUI also
    carries TMSY rows, so filtering by row term type alone would let it back in.
    """
    with zipfile.ZipFile(archive) as bundle:
        member = next(name for name in bundle.namelist() if name.endswith("RXNCONSO.RRF"))

        with bundle.open(member) as handle:
            qualified = {
                rxcui
                for rxcui, term_type, _ in _iter_rows(handle)
                if term_type in CONCEPT_TERM_TYPES
            }

        table: dict[str, dict[str, object]] = {}
        with bundle.open(member) as handle:
            for rxcui, term_type, name in _iter_rows(handle):
                if rxcui not in qualified or term_type not in NAME_TERM_TYPES:
                    continue
                entry = table.setdefault(rxcui, {"label": name, "aliases": []})
                aliases: list[str] = entry["aliases"]  # type: ignore[assignment]
                if term_type in LABEL_TERM_TYPES and len(name) < len(str(entry["label"])):
                    aliases.append(str(entry["label"]))
                    entry["label"] = name
                elif name != entry["label"] and name not in aliases:
                    aliases.append(name)
    return table


def write_terminology_tsv(table: dict[str, dict[str, object]], destination: Path) -> int:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["code", "label", "aliases"])
        for code in sorted(table, key=lambda value: int(value) if value.isdigit() else 0):
            entry = table[code]
            writer.writerow([code, entry["label"], "|".join(entry["aliases"])])  # type: ignore[arg-type]
    return len(table)


def build(archive: Path, destination: Path) -> int:
    return write_terminology_tsv(build_alias_table(archive), destination)

In [ ]:
%%writefile /kaggle/working/medical_coder_src/medical_coder/scoring.py
"""Local reading of the Vòng 1 host metric.

    final_score = 0.3 * text_score + 0.3 * assertions_score + 0.4 * candidates_score

This is *our reading* of the published formula, not the official grader. It is
reconciled against the only two public data points we have:

* our submission 01 (14.4255 from WER 83.5952 / J_assert 20.1874 / J_cand 8.6197)
* the reference solution's 27.8786 (32.1820 / 35.2687 / 19.1084)

Both reproduce exactly under `0.3*text + 0.3*assert + 0.4*cand`, which confirms
the published `WER` field is an *error rate* and `text_score = 1 - WER`.

Choices the organisers left unspecified are all localised here so a single edit
switches convention when the official scorer is released:

* A prediction matches a ground-truth concept iff **same type** and **overlapping
  character span**, matched greedily by largest overlap. A right-text/wrong-type
  prediction therefore cannot match its twin: it is a brand-new concept scoring 0
  everywhere, which is exactly the double penalty described in the rules.
* An unmatched ("spurious") prediction is counted **twice** in every applicable
  denominator. That is the mechanism that makes over-emission so expensive, and
  it is the single most important property to optimise against.
* Aggregation is global over concepts, not a mean of per-record means.

Scoring the ground truth against itself returns 1.0.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Mapping, Sequence

from .models import EntityType

WEIGHTS = {"text": 0.3, "assertions": 0.3, "candidates": 0.4}

ASSERTABLE_TYPES = {
    EntityType.SYMPTOM.value,
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}
CANDIDATE_TYPES = {
    EntityType.DIAGNOSIS.value,
    EntityType.MEDICATION.value,
}


def jaccard(prediction: set[str], truth: set[str]) -> float:
    """Jaccard with the host edge cases: both empty -> 1, one empty -> 0."""
    if not prediction and not truth:
        return 1.0
    if not prediction or not truth:
        return 0.0
    return len(prediction & truth) / len(prediction | truth)


def word_error_rate(reference: str, hypothesis: str) -> float:
    """Word-level Levenshtein error rate, (S + D + I) / N."""
    ref = reference.split()
    hyp = hypothesis.split()
    if not ref:
        return 0.0 if not hyp else 1.0
    previous = list(range(len(hyp) + 1))
    for i, ref_word in enumerate(ref, start=1):
        current = [i]
        for j, hyp_word in enumerate(hyp, start=1):
            cost = 0 if ref_word == hyp_word else 1
            current.append(
                min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + cost)
            )
        previous = current
    return previous[len(hyp)] / len(ref)


def _span(concept: Mapping) -> tuple[int, int]:
    start, end = concept["position"]
    return int(start), int(end)


def _overlap(left: tuple[int, int], right: tuple[int, int]) -> int:
    return max(0, min(left[1], right[1]) - max(left[0], right[0]))


def match_concepts(
    predictions: Sequence[Mapping],
    truth: Sequence[Mapping],
) -> tuple[dict[int, int], list[int], list[int]]:
    """Greedily pair prediction -> truth on (same type, largest overlap)."""
    pairs = []
    for pi, prediction in enumerate(predictions):
        prediction_span = _span(prediction)
        for ti, gold in enumerate(truth):
            if gold.get("type") != prediction.get("type"):
                continue
            overlap = _overlap(prediction_span, _span(gold))
            if overlap > 0:
                pairs.append((overlap, pi, ti))
    pairs.sort(key=lambda item: (-item[0], item[1], item[2]))

    matched: dict[int, int] = {}
    used_predictions: set[int] = set()
    used_truth: set[int] = set()
    for _, pi, ti in pairs:
        if pi in used_predictions or ti in used_truth:
            continue
        matched[pi] = ti
        used_predictions.add(pi)
        used_truth.add(ti)
    unmatched_predictions = [i for i in range(len(predictions)) if i not in used_predictions]
    unmatched_truth = [i for i in range(len(truth)) if i not in used_truth]
    return matched, unmatched_predictions, unmatched_truth


@dataclass
class RecordScore:
    record_id: str
    text: float
    assertions: float
    candidates: float
    predicted: int
    expected: int


@dataclass
class Score:
    text_score: float
    assertions_score: float
    candidates_score: float
    final_score: float
    per_record: list[RecordScore] = field(default_factory=list)

    def format_report(self) -> str:
        return (
            f"text       {self.text_score * 100:8.4f}\n"
            f"assertions {self.assertions_score * 100:8.4f}\n"
            f"candidates {self.candidates_score * 100:8.4f}\n"
            f"final      {self.final_score * 100:8.4f}"
        )


def score_corpus(
    predictions: Mapping[str, list],
    truth: Mapping[str, list],
    record_ids: Sequence[str] | None = None,
) -> Score:
    """Score predictions against ground truth using the host formula.

    text        = sum_matched (1 - WER) / (n_truth + 2 * n_spurious)
    assertions  = sum_matched J_assert / (n_truth_assertable + 2 * n_spurious_assertable)
    candidates  = sum_matched J_cand * w / (sum_truth w + 2 * sum_spurious w),
                  where w = len(truth candidates) + 1
    """
    if record_ids is None:
        record_ids = sorted(
            truth,
            key=lambda value: (0, int(value)) if value.isdigit() else (1, value),
        )

    text_numerator = text_denominator = 0.0
    assertion_numerator = assertion_denominator = 0.0
    candidate_numerator = candidate_denominator = 0.0
    per_record: list[RecordScore] = []

    for record_id in record_ids:
        gold = truth[record_id]
        predicted = predictions.get(record_id, [])
        matched, spurious, missed = match_concepts(predicted, gold)

        record_text = sum(
            max(0.0, 1.0 - word_error_rate(gold[ti]["text"], predicted[pi]["text"]))
            for pi, ti in matched.items()
        )
        record_text_denominator = len(gold) + 2 * len(spurious)
        text_numerator += record_text
        text_denominator += record_text_denominator

        assertable_gold = [
            i for i in range(len(gold)) if gold[i].get("type") in ASSERTABLE_TYPES
        ]
        assertable_spurious = [
            i for i in spurious if predicted[i].get("type") in ASSERTABLE_TYPES
        ]
        record_assertions = sum(
            jaccard(
                set(predicted[pi].get("assertions") or []),
                set(gold[ti].get("assertions") or []),
            )
            for pi, ti in matched.items()
            if gold[ti].get("type") in ASSERTABLE_TYPES
        )
        record_assertion_denominator = len(assertable_gold) + 2 * len(assertable_spurious)
        assertion_numerator += record_assertions
        assertion_denominator += record_assertion_denominator

        record_candidates = 0.0
        record_candidate_denominator = 0.0
        for pi, ti in matched.items():
            if gold[ti].get("type") not in CANDIDATE_TYPES:
                continue
            weight = len(gold[ti].get("candidates") or []) + 1
            record_candidates += weight * jaccard(
                set(predicted[pi].get("candidates") or []),
                set(gold[ti].get("candidates") or []),
            )
            record_candidate_denominator += weight
        for ti in missed:
            if gold[ti].get("type") in CANDIDATE_TYPES:
                record_candidate_denominator += len(gold[ti].get("candidates") or []) + 1
        for pi in spurious:
            if predicted[pi].get("type") in CANDIDATE_TYPES:
                record_candidate_denominator += 2 * (
                    len(predicted[pi].get("candidates") or []) + 1
                )
        candidate_numerator += record_candidates
        candidate_denominator += record_candidate_denominator

        per_record.append(
            RecordScore(
                record_id=record_id,
                text=record_text / record_text_denominator if record_text_denominator else 1.0,
                assertions=(
                    record_assertions / record_assertion_denominator
                    if record_assertion_denominator
                    else 1.0
                ),
                candidates=(
                    record_candidates / record_candidate_denominator
                    if record_candidate_denominator
                    else 1.0
                ),
                predicted=len(predicted),
                expected=len(gold),
            )
        )

    text = text_numerator / text_denominator if text_denominator else 1.0
    assertions = (
        assertion_numerator / assertion_denominator if assertion_denominator else 1.0
    )
    candidates = (
        candidate_numerator / candidate_denominator if candidate_denominator else 1.0
    )
    final = (
        WEIGHTS["text"] * text
        + WEIGHTS["assertions"] * assertions
        + WEIGHTS["candidates"] * candidates
    )
    return Score(text, assertions, candidates, final, per_record)


def load_records(directory: Path) -> dict[str, list]:
    return {
        path.stem: json.loads(path.read_text(encoding="utf-8"))
        for path in directory.glob("*.json")
        if path.stem.isdigit()
    }

In [ ]:
IMPORT_DIR = Path("/kaggle/working/medical_coder_src")
if str(IMPORT_DIR) not in sys.path:
    sys.path.insert(0, str(IMPORT_DIR))

modules = sorted(p.name for p in (IMPORT_DIR / "medical_coder").glob("*.py"))
print(f"{len(modules)} module ->", IMPORT_DIR)
print(" ", ", ".join(modules))

import medical_coder
from medical_coder import exact_link, gliner_ner, pipeline_v2, selector, submission
print("\nmedical_coder:", medical_coder.__file__)

REPO = None   # không có repo trên đĩa; mọi thứ dựng ra nằm ở /kaggle/working

## 4. Dữ liệu đầu vào

Ưu tiên Dataset đã attach. Nếu không có Dataset nào, notebook dùng bản test Vòng 1
**nhúng sẵn** bên dưới — nhờ vậy chỉ cần tải lên đúng một tệp notebook, không cần
attach gì cả.

> Khi chấm trên private test, Ban Tổ chức sẽ cấp input khác. Lúc đó **phải** attach
> Dataset input mới; cell này sẽ tự ưu tiên nó và in rõ nguồn đang dùng, nhưng nếu
> quên attach thì nó rơi về bản public test nhúng sẵn và điểm sẽ sai. Hãy đọc dòng
> `nguồn input:` mà cell in ra.

In [ ]:
# tar.gz của input/*.txt (public test Vòng 1). Sinh tự động — đừng sửa tay.
INPUT_TGZ_B64 = (
    "..."
    "UAUA"
)
print("blob input:", len(INPUT_TGZ_B64), "ký tự base64")

In [ ]:
# Đường dẫn Dataset đã biết, thử trước để khỏi quét toàn bộ /kaggle/input.
KNOWN_INPUT_DIRS = [
    Path("/kaggle/input/datasets/thanhhiepvo/viettelairace/input"),
    Path("/kaggle/input/viettelairace/input"),
    Path("/kaggle/input/viettelairace"),
]

def is_input_dir(folder):
    return folder.is_dir() and (folder / "1.txt").exists() and (folder / "100.txt").exists()

def find_attached_input():
    for folder in KNOWN_INPUT_DIRS:
        if is_input_dir(folder):
            return folder
    base = Path("/kaggle/input")
    if base.exists():
        for path in base.rglob("1.txt"):
            if is_input_dir(path.parent):
                return path.parent
    return None

INPUT_DIR = find_attached_input()
INPUT_SOURCE = "Dataset đã attach"

if INPUT_DIR is None:
    INPUT_SOURCE = "BẢN NHÚNG trong notebook (public test Vòng 1)"
    target = WORK / "input_embedded"
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True)
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(INPUT_TGZ_B64)), mode="r:gz") as tar:
        for member in tar.getmembers():
            if member.name.startswith("/") or ".." in Path(member.name).parts:
                raise SystemExit(f"tarball có đường dẫn không hợp lệ: {member.name}")
        try:
            tar.extractall(target, filter="data")
        except TypeError:
            tar.extractall(target)
    INPUT_DIR = target / "input"

n = len(list(INPUT_DIR.glob("*.txt")))
print("nguồn input:", INPUT_SOURCE)
print("đường dẫn  :", INPUT_DIR, f"({n} tệp)")
assert n == 100, f"Cần đúng 100 tệp, thấy {n}"
if INPUT_DIR == WORK / "input_embedded" / "input":
    print("\n>>> Đang dùng public test nhúng sẵn. Nếu đây là lần chạy cho PRIVATE TEST,")
    print(">>> hãy attach Dataset input của BTC rồi chạy lại cell này. <<<")

## 5. Knowledge base

* **ICD-10 tiếng Việt** — dựng từ Phụ lục TT06/2026/TT-BYT (Bộ Y tế) và **nhúng
  sẵn** trong notebook. Đây là thay đổi quan trọng nhất ở phần gán mã: KB cũ là
  bản tiếng Anh của CDC nên một mention như `viêm túi mật` không thể khớp alias
  nào.
* **RxNorm** — Current Prescribable Content của NLM. Không nhúng được vì là dữ
  liệu của bên thứ ba, nên cell dưới tự tải khi bật Internet. Thiếu nó thì
  candidates `THUỐC` rỗng, pipeline vẫn chạy.

In [ ]:
# data/terminology/icd10_vn.tsv nén gzip rồi base64. Sinh tự động — đừng sửa tay.
ICD_TSV_GZ_B64 = (
    "..."
    "//8AsL3CEsjHEwA="
)
print("blob ICD:", len(ICD_TSV_GZ_B64), "ký tự base64")

In [ ]:
import gzip

# /kaggle/input là READ-ONLY, nên mọi thứ dựng ra phải nằm ở /kaggle/working.
TERM_DIR = WORK / "terminology"
TERM_DIR.mkdir(parents=True, exist_ok=True)
ICD_TSV = TERM_DIR / "icd10_vn.tsv"

ICD_TSV.write_bytes(gzip.decompress(base64.b64decode(ICD_TSV_GZ_B64)))

rows = ICD_TSV.read_text(encoding="utf-8").splitlines()
print(f"ICD KB: {len(rows) - 1:,} mã -> {ICD_TSV} ({ICD_TSV.stat().st_size:,} bytes)")
print("ví dụ:", rows[1])

In [ ]:
from medical_coder.rxnorm_kb import build as build_rxnorm

RX_TSV = TERM_DIR / "rxnorm.tsv"
RX_URL = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_07062026.zip"

def find_rxnorm_archive():
    for base in (Path("/kaggle/input"), WORK):
        if base.exists():
            for path in base.rglob("RxNorm_full_prescribe_*.zip"):
                return path
    return None

if RX_TSV.exists():
    print("dùng RxNorm TSV có sẵn:", RX_TSV)
else:
    archive = find_rxnorm_archive()
    if archive is None:
        archive = WORK / "rxnorm.zip"
        print("tải RxNorm …")
        rc = subprocess.run(["curl", "-sSL", "--max-time", "600", "-o", str(archive), RX_URL],
                            check=False).returncode
        if rc != 0 or not archive.exists() or archive.stat().st_size < 10_000_000:
            print("CẢNH BÁO: tải RxNorm thất bại — candidates THUỐC sẽ rỗng")
            archive = None
    if archive is not None:
        print("dựng RxNorm KB từ:", archive)
        print("số RxCUI:", build_rxnorm(archive, RX_TSV))
    else:
        RX_TSV = None

## 6. Weights

| Model | Vai trò | Tham số |
|---|---|---:|
| `urchade/gliner_multi-v2.1` | NER | 0.289B |
| `Qwen/Qwen3-4B-Instruct-2507` | corrector (teacher chính) | 4.022B |
| `Qwen/Qwen3.5-4B` | teacher phụ, chỉ dùng cho additions | 4.206B |
| **Tổng** | | **8.517B** < 9B |

Teacher phụ là **tuỳ chọn**. Nếu tải không được thì pipeline vẫn chạy với riêng
corrector (tổng 4.311B) và bỏ qua bước additions — vì additions bắt buộc phải có
hai teacher đồng thuận.

Nếu chỉ muốn một teacher mạnh hơn: đặt `PRIMARY = "Qwen/Qwen3-8B"` và
`SECONDARY = None` → tổng 8.489B, vẫn dưới 9B.

In [ ]:
GLINER_MODEL = "urchade/gliner_multi-v2.1"
PRIMARY   = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY = "Qwen/Qwen3.5-4B"      # đặt None để chỉ chạy corrector

TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for name in ("HF_TOKEN", "HF_KEY", "HUGGINGFACE_TOKEN", "HUGGINGFACE_KEY"):
        try:
            TOKEN = UserSecretsClient().get_secret(name)
            if TOKEN:
                print("dùng secret:", name)
                break
        except Exception:
            continue
except ImportError:
    pass

MODEL_DIR = WORK / "models"
MODEL_DIR.mkdir(exist_ok=True)

def resolve_model(repo_id):
    """Trả về đường dẫn local; ưu tiên dataset đã attach, sau đó mới tải."""
    leaf = repo_id.split("/")[-1]
    for base in (Path("/kaggle/input"), MODEL_DIR):
        if base.exists():
            for path in base.rglob(leaf):
                if path.is_dir() and any(path.glob("config.json")):
                    return str(path)
    from huggingface_hub import snapshot_download
    target = MODEL_DIR / leaf
    snapshot_download(repo_id=repo_id, local_dir=str(target), token=TOKEN,
                      ignore_patterns=["*.pth", "*.onnx", "*.msgpack", "*.h5"])
    return str(target)

GLINER_PATH = resolve_model(GLINER_MODEL)
print("gliner:", GLINER_PATH)

PRIMARY_PATH = resolve_model(PRIMARY)
print("primary:", PRIMARY_PATH)

SECONDARY_PATH = None
if SECONDARY:
    try:
        SECONDARY_PATH = resolve_model(SECONDARY)
        print("secondary:", SECONDARY_PATH)
    except Exception as exc:
        print(f"không tải được teacher phụ ({exc}) — chỉ chạy corrector, bỏ additions")

In [ ]:
# Sau bước provision, khoá offline: inference hoàn toàn self-host, không gọi API.
TOKEN = None
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("đã khoá chế độ offline")

## 7. Cấu hình

In [ ]:
import torch
from medical_coder.gliner_ner import DEFAULT_THRESHOLDS
from medical_coder.models import EntityType
from medical_coder.pipeline_v2 import PipelineV2Config, run_pipeline_v2

HAS_CUDA = torch.cuda.is_available()
NGPU = torch.cuda.device_count() if HAS_CUDA else 0

OUTPUT_DIR = WORK / "output"
ZIP_PATH   = WORK / "output.zip"

# Ngưỡng theo từng type. GLiNER có phân bố score khác nhau theo label nên một
# ngưỡng chung là sai; các giá trị này lấy từ lời giải tham chiếu 27.8786.
THRESHOLDS = dict(DEFAULT_THRESHOLDS)
for k, v in THRESHOLDS.items():
    print(f"  {k.value:22s} {v}")

CONFIG = dict(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_path=GLINER_PATH,
    device="cuda" if HAS_CUDA else "cpu",
    icd_kb=ICD_TSV,
    rxnorm_kb=RX_TSV,
    thresholds=THRESHOLDS,
    max_candidates=1,          # >1 làm phình mẫu số candidate
    primary_teacher=PRIMARY_PATH if HAS_CUDA else None,
    secondary_teacher=SECONDARY_PATH if HAS_CUDA else None,
    teacher_device="cuda:0" if HAS_CUDA else "cpu",
    teacher_quantization="4bit",
    teacher_batch_size=48 if NGPU else 8,
)
print("\nGPU:", NGPU, "| corrector:", bool(CONFIG["primary_teacher"]),
      "| additions:", bool(CONFIG["secondary_teacher"]))

## 8. Smoke test (2 bản ghi)

Chạy thử trước khi chạy đủ 100 để bắt lỗi cấu hình sớm. Bước này **không** tạo ZIP.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

SMOKE_DIR = WORK / "output_smoke"
smoke = run_pipeline_v2(PipelineV2Config(**{**CONFIG, "output_dir": SMOKE_DIR,
                                            "selected_ids": frozenset({"1", "2"})}))
print("\nsmoke concepts:", smoke)

for stem in ("1", "2"):
    data = json.loads((SMOKE_DIR / f"{stem}.json").read_text(encoding="utf-8"))
    raw = (INPUT_DIR / f"{stem}.txt").read_text(encoding="utf-8")
    assert all(raw[c["position"][0]:c["position"][1]] == c["text"] for c in data), "offset sai"
    print(f"\n--- {stem}.json ({len(data)} concept) ---")
    for c in data[:6]:
        print(f"  {c['position']} {c['type']:20s} {c['text'][:44]!r} {c.get('candidates', '')}")
print("\noffset khớp nguyên văn trên cả hai bản ghi")

## 9. Chạy đủ 100 bản ghi

In [ ]:
import time

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

start = time.time()
total = run_pipeline_v2(PipelineV2Config(**CONFIG))
print(f"\n{total} concept trong {time.time() - start:.0f}s")

## 10. Kiểm tra và đóng gói

In [ ]:
from medical_coder.submission import create_submission_zip, validate_all

validate_all(INPUT_DIR, OUTPUT_DIR)
print("validator: PASS")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
create_submission_zip(OUTPUT_DIR, ZIP_PATH)

import zipfile, hashlib
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
    assert names == [f"output/{i}.json" for i in range(1, 101)], "cấu trúc ZIP sai"
    assert archive.testzip() is None, "ZIP hỏng"
print(f"ZIP OK: {len(names)} tệp, {ZIP_PATH.stat().st_size:,} bytes")
print("sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())

In [ ]:
from collections import Counter

records = {p.stem: json.loads(p.read_text(encoding="utf-8"))
           for p in OUTPUT_DIR.glob("*.json") if p.stem.isdigit()}
concepts = [c for v in records.values() for c in v]
types = Counter(c["type"] for c in concepts)
with_codes = [c for c in concepts if c.get("candidates")]

print(f"tổng concept        : {len(concepts)}")
print(f"trung bình / bản ghi: {len(concepts) / len(records):.2f}")
print(f"bản ghi rỗng        : {[k for k, v in records.items() if not v]}")
for k in ("TRIỆU_CHỨNG", "CHẨN_ĐOÁN", "THUỐC", "TÊN_XÉT_NGHIỆM", "KẾT_QUẢ_XÉT_NGHIỆM"):
    print(f"  {k:22s} {types.get(k, 0)}")
print(f"concept có candidate: {len(with_codes)}")
print(f"tổng mã xuất ra     : {sum(len(c['candidates']) for c in with_codes)}")
print(f"nhãn assertion      : {sum(len(c['assertions']) for c in concepts)} (chủ ý để rỗng)")

## 11. Tải kết quả

`/kaggle/working/output.zip` — nộp trực tiếp tệp này.

In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(ZIP_PATH)))

## 12. Ghi chú

**Đã cố ý bỏ:**

* **Assertions để rỗng.** Lời giải tham chiếu đo được `isNegated` tách biệt ở AUC
  0.497 (ngang ngẫu nhiên); mọi rule đều emit thừa. Một assertion sai mất trọn
  Jaccard của concept đó, trong khi dự đoán rỗng đúng với ground truth rỗng được
  1.0. Chỉ nên bật lại khi đã có dữ liệu gán nhãn.
* **Candidate tối đa 1 và chỉ khi alias khớp duy nhất.** Bỏ toàn bộ candidate chỉ
  làm candidate Jaccard của họ giảm 0.0036 — thành phần 40% này gần như hoàn toàn
  do chất lượng khớp concept quyết định, không phải do tra đúng mã.
* **Additions chỉ cho type không có candidate.** Thêm nhầm một CHẨN_ĐOÁN/THUỐC
  còn bị tính vào mẫu số candidate.

**Chưa kiểm chứng:** chưa có ground truth nên chưa đo được điểm cục bộ. Sau khi
gán nhãn 15–20 bản ghi, dùng:

```bash
medical-coder score --output-dir output --truth-dir data/labelled --per-record
```

Scorer tự chấm ground truth bằng 1.0 và tái lập đúng cả hai mốc điểm đã công bố
(14.4255 và 27.8786).